# OPV Closed-Loop Bayesian Optimization Campaign

This notebook demonstrates a restartable ask-tell Bayesian-optimization campaign with `matgpr`. It uses the OPV dataset retrospectively so we can simulate a closed loop: recommend candidates, choose a subset for experiments, reveal withheld measurements, log observations, restart the campaign, and prepare the next iteration.

The notebook focuses on campaign bookkeeping and reproducibility. It pairs naturally with `opv_bo_recommendation_audit.ipynb`, which explains why individual candidates were recommended.

## 1. Setup

The campaign log is written to a temporary directory during notebook execution. That keeps the example fully executable without committing generated CSV files.

In [ ]:
from __future__ import annotations

import os
import sys
import tempfile
from pathlib import Path

cache_root = Path(tempfile.gettempdir()) / "matgpr_notebook_cache"
os.environ.setdefault("MPLCONFIGDIR", str(cache_root / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(cache_root / "xdg"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "matgpr").exists():
            return candidate
        sibling = candidate / "matgpr"
        if (sibling / "pyproject.toml").exists() and (sibling / "matgpr").exists():
            return sibling
    raise RuntimeError("Could not find the matgpr project root")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from matgpr import (
    CandidateDuplicatePolicy,
    fit_botorch_surrogate,
    log_bo_recommendations,
    log_observations,
    log_selected_experiments,
    plot_bo_campaign_progress,
    rank_discrete_candidates,
    resume_bo_campaign,
    select_diverse_batch,
    summarize_closed_loop_log,
)

RANDOM_STATE = 43
CAMPAIGN_ID = "opv_closed_loop_demo"
INITIAL_MEASURED_COUNT = 28
CANDIDATE_COUNT = 110
TOP_K = 6
SELECTED_PER_ITERATION = 3
BO_FIT_MODEL = True
TARGET_COLUMN = "PCE"

BASE_FEATURE_COLUMNS = [
    "polarizability",
    "delLA",
    "delLD",
    "N_atom",
    "Eg",
    "lamda_h",
    "DIP",
    "AL-DH",
    "delHD",
    "E_bind",
    "DL-AL",
    "delGE",
    "E_T1",
]
PHYSICS_COLUMNS = ["physics_degeneracy_score", "physics_binding_score"]
DIVERSITY_COLUMNS = ["Eg", "E_bind", "delHD", "delLD", "delLA", *PHYSICS_COLUMNS]

plt.rcParams.update({
    "figure.dpi": 140,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## 2. Load Measured And Candidate OPV Data

The full OPV dataset is known, but the campaign treats most rows as unmeasured candidates. The hidden target values are revealed only when the simulated experiment returns an observation.

In [ ]:
def load_opv_data() -> pd.DataFrame:
    data_path = PROJECT_ROOT / "examples" / "opv" / "dataset.pkl"
    data = pd.read_pickle(data_path)
    data = data.rename(columns={"#Sno.": "candidate_id"})
    data["candidate_id"] = "opv_" + data["candidate_id"].astype(str)
    keep_columns = ["candidate_id", TARGET_COLUMN, *BASE_FEATURE_COLUMNS]
    return data.loc[:, keep_columns].dropna().reset_index(drop=True)


def add_physics_scores(frame: pd.DataFrame, reference: pd.DataFrame) -> pd.DataFrame:
    result = frame.loc[:, BASE_FEATURE_COLUMNS].astype(float).copy()
    physics_columns = ["delHD", "delLD", "delLA", "E_bind"]
    physics_source = frame.loc[:, physics_columns].astype(float)
    reference_physics = reference.loc[:, physics_columns].astype(float)
    reference_mean = reference_physics.mean(axis=0)
    reference_std = reference_physics.std(axis=0, ddof=0).replace(0.0, 1.0)
    z_scores = (physics_source - reference_mean) / reference_std
    result["physics_degeneracy_score"] = -(
        z_scores["delHD"] + z_scores["delLD"] + z_scores["delLA"]
    ) / 3.0
    result["physics_binding_score"] = -z_scores["E_bind"]
    return result


opv_data = load_opv_data()
initial_measured = opv_data.sample(n=INITIAL_MEASURED_COUNT, random_state=RANDOM_STATE)
initial_measured = initial_measured.reset_index(drop=True)
initial_candidates = opv_data.loc[~opv_data["candidate_id"].isin(initial_measured["candidate_id"])]
initial_candidates = initial_candidates.sample(n=CANDIDATE_COUNT, random_state=RANDOM_STATE + 1)
initial_candidates = initial_candidates.reset_index(drop=True)

print(f"Initial measured rows: {len(initial_measured)}")
print(f"Initial candidate rows: {len(initial_candidates)}")

## 3. Helper Functions For One Ask-Tell Iteration

`run_bo_ask` creates a recommendation table from the current measured data and available candidate pool. BoTorch fits the surrogate and ranks the pool with upper confidence bound.

In [ ]:
def build_feature_frames(
    measured: pd.DataFrame,
    candidates: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    X_train_raw = add_physics_scores(measured, measured)
    X_candidates_raw = add_physics_scores(candidates, measured)
    scaler = StandardScaler().fit(X_train_raw)
    X_train = pd.DataFrame(scaler.transform(X_train_raw), columns=X_train_raw.columns)
    X_candidates = pd.DataFrame(scaler.transform(X_candidates_raw), columns=X_candidates_raw.columns)
    candidate_metadata = candidates.loc[:, ["candidate_id", TARGET_COLUMN, "Eg", "E_bind", "delHD", "delLD", "delLA"]]
    candidate_metadata = candidate_metadata.rename(columns={TARGET_COLUMN: "withheld_pce_for_retrospective"})
    candidate_metadata = pd.concat(
        [
            candidate_metadata.reset_index(drop=True),
            X_candidates_raw.loc[:, PHYSICS_COLUMNS].reset_index(drop=True),
        ],
        axis=1,
    )
    return X_train, X_candidates, candidate_metadata


def run_bo_ask(
    measured: pd.DataFrame,
    available_candidates: pd.DataFrame,
    unavailable_candidates: pd.DataFrame,
    *,
    top_k: int,
) -> tuple[pd.DataFrame, str]:
    X_train, X_candidates, candidate_metadata = build_feature_frames(measured, available_candidates)
    duplicate_policy = None
    if not unavailable_candidates.empty:
        duplicate_policy = CandidateDuplicatePolicy(
            existing_candidates=unavailable_candidates,
            key_columns=("candidate_id",),
        )

    surrogate = fit_botorch_surrogate(
        X_train,
        measured[TARGET_COLUMN],
        maximize=True,
        normalize_features=True,
        standardize_target=True,
        fit_model=BO_FIT_MODEL,
    )
    ranked = rank_discrete_candidates(
        surrogate,
        X_candidates,
        acquisition_function="upper_confidence_bound",
        beta=0.35,
        candidate_data=candidate_metadata,
        duplicate_policy=duplicate_policy,
        duplicate_policy_action="filter",
    )
    source = "BoTorch upper confidence bound"

    selected = select_diverse_batch(
        ranked,
        top_k=min(top_k, len(ranked)),
        score_column="matgpr_acquisition",
        feature_columns=DIVERSITY_COLUMNS,
        diversity_weight=0.2,
        return_all=False,
    )
    return selected.reset_index(drop=True), source


def reveal_observations(selected: pd.DataFrame, iteration: int) -> pd.DataFrame:
    observations = selected.loc[:, ["candidate_id", "withheld_pce_for_retrospective"]].copy()
    observations = observations.rename(columns={"withheld_pce_for_retrospective": TARGET_COLUMN})
    observations["reported_std"] = 0.20
    observations["source"] = f"retrospective_iteration_{iteration}"
    return observations

## 4. Start Or Resume The Campaign

A real campaign would reuse a persistent CSV path. This example starts with a new temporary path, then calls `resume_bo_campaign` before each ask step to show the restart pattern.

In [ ]:
campaign_dir = Path(tempfile.mkdtemp(prefix="matgpr_opv_bo_"))
log_path = campaign_dir / "opv_closed_loop_campaign.csv"

state = resume_bo_campaign(
    log_path,
    campaign_id=CAMPAIGN_ID,
    key_columns=("candidate_id",),
    candidate_pool=initial_candidates,
)

print(f"Campaign log path: {log_path}")
print(f"Next iteration: {state.next_iteration}")
print(f"Available candidates: {len(state.available_candidates)}")
print(f"Pending experiments: {len(state.pending_experiments)}")
print(f"Completed observations: {len(state.completed_experiments)}")

## 5. Iteration 0: Ask, Select, Tell

The ask step recommends a diverse batch. The tell step logs selected experiments and observed PCE values. The logged observation rows use iteration `next_iteration + 1`, meaning they are available for the next model update.

In [ ]:
iteration = state.next_iteration
recommendations_0, ranking_source_0 = run_bo_ask(
    initial_measured,
    state.available_candidates,
    state.unavailable_candidates,
    top_k=TOP_K,
)
selected_0 = recommendations_0.head(SELECTED_PER_ITERATION).copy()
observations_0 = reveal_observations(selected_0, iteration=iteration)

log_bo_recommendations(
    recommendations_0,
    path=log_path,
    campaign_id=CAMPAIGN_ID,
    iteration=iteration,
    model_name="opv_gpr",
    acquisition_function=ranking_source_0,
    metadata={"example": "closed_loop_opv"},
    timestamp="2026-06-30T12:00:00+00:00",
)
log_selected_experiments(
    selected_0,
    path=log_path,
    campaign_id=CAMPAIGN_ID,
    iteration=iteration,
    selection_policy="diverse_top_batch",
    timestamp="2026-06-30T12:05:00+00:00",
)
log_observations(
    observations_0,
    path=log_path,
    campaign_id=CAMPAIGN_ID,
    iteration=iteration + 1,
    target_column=TARGET_COLUMN,
    metadata={"measurement_mode": "retrospective_reveal"},
    timestamp="2026-06-30T12:10:00+00:00",
)

display(selected_0.loc[:, ["candidate_id", "matgpr_rank", "matgpr_acquisition", "matgpr_predicted_mean", "matgpr_predicted_std"]])
display(observations_0)

## 6. Restart State After Iteration 0

After logging observations, `resume_bo_campaign` identifies completed and unavailable candidates and returns the remaining available pool for the next ask step.

In [ ]:
state_after_0 = resume_bo_campaign(
    log_path,
    campaign_id=CAMPAIGN_ID,
    key_columns=("candidate_id",),
    candidate_pool=initial_candidates,
)

print(f"Current iteration in log: {state_after_0.current_iteration}")
print(f"Last recommendation iteration: {state_after_0.last_recommendation_iteration}")
print(f"Next ask iteration: {state_after_0.next_iteration}")
print(f"Available candidates: {len(state_after_0.available_candidates)}")
print(f"Unavailable candidates: {len(state_after_0.unavailable_candidates)}")
print(f"Pending experiments: {len(state_after_0.pending_experiments)}")
print(f"Completed observations: {len(state_after_0.completed_experiments)}")

## 7. Iteration 1: Update Measured Data And Ask Again

The newly observed rows are appended to the measured set before the next recommendation. The restart state prevents already completed or pending candidates from being recommended again.

In [ ]:
measured_after_0 = pd.concat(
    [
        initial_measured,
        opv_data.loc[opv_data["candidate_id"].isin(observations_0["candidate_id"])],
    ],
    ignore_index=True,
)

iteration = state_after_0.next_iteration
recommendations_1, ranking_source_1 = run_bo_ask(
    measured_after_0,
    state_after_0.available_candidates,
    state_after_0.unavailable_candidates,
    top_k=TOP_K,
)
selected_1 = recommendations_1.head(SELECTED_PER_ITERATION).copy()
observations_1 = reveal_observations(selected_1, iteration=iteration)

log_bo_recommendations(
    recommendations_1,
    path=log_path,
    campaign_id=CAMPAIGN_ID,
    iteration=iteration,
    model_name="opv_gpr",
    acquisition_function=ranking_source_1,
    metadata={"example": "closed_loop_opv"},
    timestamp="2026-06-30T13:00:00+00:00",
)
log_selected_experiments(
    selected_1,
    path=log_path,
    campaign_id=CAMPAIGN_ID,
    iteration=iteration,
    selection_policy="diverse_top_batch",
    timestamp="2026-06-30T13:05:00+00:00",
)
log_observations(
    observations_1,
    path=log_path,
    campaign_id=CAMPAIGN_ID,
    iteration=iteration + 1,
    target_column=TARGET_COLUMN,
    metadata={"measurement_mode": "retrospective_reveal"},
    timestamp="2026-06-30T13:10:00+00:00",
)

display(selected_1.loc[:, ["candidate_id", "matgpr_rank", "matgpr_acquisition", "matgpr_predicted_mean", "matgpr_predicted_std"]])
display(observations_1)

## 8. Summarize And Visualize The Campaign Log

The campaign log contains recommendation, selection, and observation records. The summary table and progress plot make it easy to audit whether each iteration moved from ask to selected experiments to measured outcomes.

In [ ]:
campaign_log = pd.read_csv(log_path)
campaign_summary = summarize_closed_loop_log(
    campaign_log,
    campaign_id=CAMPAIGN_ID,
    target_column=TARGET_COLUMN,
)

display(campaign_summary)

fig, ax, progress_summary = plot_bo_campaign_progress(
    campaign_log,
    campaign_id=CAMPAIGN_ID,
    target_column=TARGET_COLUMN,
)
ax.set_title("Closed-loop OPV BO campaign")
plt.show()

display(progress_summary)

## 9. Final Restart Check

This is the key practical step for a real campaign. At the beginning of the next work session, the user can call `resume_bo_campaign` with the saved CSV log and the full candidate pool to recover the next iteration, pending/completed experiments, and the available candidate pool.

In [ ]:
final_state = resume_bo_campaign(
    log_path,
    campaign_id=CAMPAIGN_ID,
    key_columns=("candidate_id",),
    candidate_pool=initial_candidates,
)

print(f"Next ask iteration: {final_state.next_iteration}")
print(f"Available candidates for next ask: {len(final_state.available_candidates)}")
print(f"Completed experiments: {len(final_state.completed_experiments)}")
print(f"Pending experiments: {len(final_state.pending_experiments)}")

final_state.completed_experiments.loc[:, ["candidate_id", TARGET_COLUMN, "reported_std", "source"]].head(10)

## 10. Takeaways

A reliable BO campaign needs more than candidate ranking. It needs a durable record of what was recommended, what was actually selected, which observations came back, and which candidates must be excluded from future asks. `matgpr` provides these utilities as lightweight CSV-based building blocks so the same workflow can later be connected to lab databases, cloud storage, or a GenMatics portal backend.